In [11]:
import pandas as pd
import json
import glob

# 1. 여러 개의 JSON 파일 경로 읽기
file_list = glob.glob("Z:/microsoft/project2/code/*.json") # 로컬 폴더 경로
rows = []

for file in file_list:
    with open(file, 'r', encoding='utf-8') as f:
        data = json.load(f)
        meta = data['metadata']
        
        # 자막이 비어있거나 에러 메시지만 있는 경우 처리
        for entry in data['transcript']:
            # metadata의 정보와 transcript의 정보를 합칩니다.
            combined = {**meta, **entry} 
            rows.append(combined)

df = pd.DataFrame(rows)

# df.to_csv("combined_data.csv", index=False)

## 1단계. 노이즈 제거 버전



In [14]:
import os
import json
import glob
import re

def ultimate_noise_cutter(text):
    """
    유튜브 자막의 노이즈를 극한으로 제거하는 함수.
    노이즈로 판별되어 버려야 할 문장인 경우 None을 반환하고,
    살려야 할 문장이면 깔끔하게 정제된 문자열을 반환합니다.
    """
    if not isinstance(text, str):
        return None

    # ==========================================
    # [1차 필터] 패턴 파괴 (정규표현식)
    # ==========================================
    # 1. 괄호와 그 안의 내용 제거: [음악], (박수 소리), <오프닝> 등
    text = re.sub(r'\[.*?\]|\(.*?\)|<.*?>', '', text)
    
    # 2. URL 웹사이트 링크 제거 (http, www 등)
    text = re.sub(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\(\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+', '', text)
    text = re.sub(r'www\.[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+', '', text)
    
    # 3. 이메일 주소 제거
    text = re.sub(r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+', '', text)
    
    # 양끝 공백 및 특수문자 찌꺼기 정리
    text = text.strip()

    # ==========================================
    # [2차 필터] 유튜브 상투어 폭격 (통째로 버리기)
    # ==========================================
    # 이 단어들이 포함된 자막은 문맥상 정보가가 "0"에 수렴하므로 가차없이 버립니다.
    spam_phrases = [
        # 한국어 스팸 패턴
        "구독과", "좋아요", "알림 설정", "시청해주셔서", "시청해 주셔서", "안녕하세요",
        "채널에 오신", "후원 계좌", "후원계좌", "링크를 확인", "댓글로 남겨", "문의는", "클릭해",
        # 영어 스팸 패턴 (올려주신 데이터 대비)
        "subscribe", "hit the bell", "thanks for watching", "welcome back", 
        "link in the description", "leave a comment", "smash that like"
    ]
    
    # 텍스트를 소문자로 변환하여 스팸 단어가 하나라도 있는지 검사
    text_lower = text.lower()
    if any(spam in text_lower for spam in spam_phrases):
        return None  # 스팸 문장이면 None을 반환하여 파이프라인에서 폐기(Drop)

    # ==========================================
    # [3차 필터] 무의미한 초소형 텍스트 소거
    # ==========================================
    # 공백과 특수문자(마침표, 쉼표 등)를 제외한 '순수 글자 수' 계산
    pure_text_length = len(re.sub(r'[\W_]+', '', text))
    
    # 순수 글자 수가 2글자 이하인 경우 (예: "네.", "아..", "So,") 폐기
    if pure_text_length <= 2:
        return None

    # 모든 필터를 무사히 통과한 '진짜 알맹이'만 반환
    return text

# ==========================================
# JSON 파일 개별 처리 및 새로 저장
# ==========================================
input_path = "Z:/microsoft/project2/code/*.json"
file_list = glob.glob(input_path)

for file_path in file_list:
    # 이미 '_cleaned'가 붙은 파일이 있다면 스킵 (중복 실행 방지)
    if "_cleaned.json" in file_path:
        continue

    try:
        # 원본 JSON 읽기
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        meta = data.get('metadata', {})
        raw_transcript = data.get('transcript', [])
        
        cleaned_transcript = []
        
        # 자막 노이즈 제거 진행
        for item in raw_transcript:
            clean_text = ultimate_noise_cutter(item.get("text", ""))
            if clean_text: 
                cleaned_transcript.append({
                    "start": item["start"],
                    "text": clean_text
                })
        
        # 새로운 JSON 구조 조립 (메타데이터 유지 + 정제된 자막)
        cleaned_data = {
            "metadata": meta,
            "transcript": cleaned_transcript
        }
        
        # ----------------------------------------------------
        # [핵심 로직] 원본 이름에서 새 이름(_cleaned) 만들기
        # ----------------------------------------------------
        dir_name = os.path.dirname(file_path)            # 폴더 경로 추출
        base_name = os.path.basename(file_path)          # 파일명 추출 (예: video1.json)
        file_name_only, ext = os.path.splitext(base_name) # 이름과 확장자 분리 (video1, .json)
        
        new_file_name = f"{file_name_only}_cleaned{ext}"  # video1_cleaned.json
        new_file_path = os.path.join(dir_name, new_file_name) # 최종 경로 병합
        
        # 새로운 JSON 파일로 저장
        # ensure_ascii=False 를 넣어야 한글이 깨지지 않고 저장됩니다.
        with open(new_file_path, 'w', encoding='utf-8') as out_f:
            json.dump(cleaned_data, out_f, ensure_ascii=False, indent=4)
            
        print(f"✅ 정제 완료: {base_name} -> {new_file_name}")
        
    except Exception as e:
        print(f"❌ 에러 발생 ({file_path}): {e}")

print("모든 파일의 전처리 및 저장이 완료되었습니다!")


✅ 정제 완료: Are perpetual motion machines possible_data.json -> Are perpetual motion machines possible_data_cleaned.json
✅ 정제 완료: Contrails vs Chemtrails What Those White Streaks Really Are_data.json -> Contrails vs Chemtrails What Those White Streaks Really Are_data_cleaned.json
✅ 정제 완료: Destroying Flat Earth Without Using Science - Part 2 The Stars_data.json -> Destroying Flat Earth Without Using Science - Part 2 The Stars_data_cleaned.json
✅ 정제 완료: Free Energy Devices Build and Science_data.json -> Free Energy Devices Build and Science_data_cleaned.json
✅ 정제 완료: Infinite Energy Generator with a Car Alternator220 Volts-10KW_data.json -> Infinite Energy Generator with a Car Alternator220 Volts-10KW_data_cleaned.json
✅ 정제 완료: NASAs Secret Technology! Free Internet Wherever You Want, No Plan_data.json -> NASAs Secret Technology! Free Internet Wherever You Want, No Plan_data_cleaned.json
✅ 정제 완료: [바로리뷰]오투스 플러스 30일 사용 후기_data.json -> [바로리뷰]오투스 플러스 30일 사용 후기_data_cleaned.json
✅ 정제 완료: [예양육각수]

In [17]:
import json
import glob
import os

# 1. 정제된(_cleaned.json) 파일 목록 불러오기
file_list = glob.glob("Z:/microsoft/project2/code/*_cleaned.json")

print(f"총 {len(file_list)}개의 파일을 분석합니다...\n")
print("-" * 60)

# 2. 파일 순회하며 영상별 '총 글자 수' 계산 및 출력
for file_path in file_list:
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            
        base_name = os.path.basename(file_path)
        
        # 파일 하나(영상 한 편)의 텍스트 총 글자 수 계산
        total_length = 0
        for item in data.get('transcript', []):
            text = item.get('text', '')
            total_length += len(text) # 공백 포함 글자 수 누적
            
        # 결과 출력 (천 단위 콤마 포함)
        print(f"📄 파일명: {base_name:<50} | 📝 총 글자 수: {total_length:,}자")
            
    except Exception as e:
        print(f"❌ 읽기 에러 발생 ({file_path}): {e}")

print("-" * 60)
print("분석이 완료되었습니다.")

총 12개의 파일을 분석합니다...

------------------------------------------------------------
📄 파일명: Are perpetual motion machines possible_data_cleaned.json | 📝 총 글자 수: 4,715자
📄 파일명: Contrails vs Chemtrails What Those White Streaks Really Are_data_cleaned.json | 📝 총 글자 수: 2,181자
📄 파일명: Destroying Flat Earth Without Using Science - Part 2 The Stars_data_cleaned.json | 📝 총 글자 수: 5,438자
📄 파일명: Free Energy Devices Build and Science_data_cleaned.json | 📝 총 글자 수: 11,489자
📄 파일명: Infinite Energy Generator with a Car Alternator220 Volts-10KW_data_cleaned.json | 📝 총 글자 수: 4,415자
📄 파일명: NASAs Secret Technology! Free Internet Wherever You Want, No Plan_data_cleaned.json | 📝 총 글자 수: 2,106자
📄 파일명: [바로리뷰]오투스 플러스 30일 사용 후기_data_cleaned.json          | 📝 총 글자 수: 1,098자
📄 파일명: [예양육각수] 육각수의 발견 - 물에서 보물을 발견하다!_data_cleaned.json  | 📝 총 글자 수: 4,610자
📄 파일명: 다누리호 내가 말했째 사기라고_data_cleaned.json                 | 📝 총 글자 수: 2,939자
📄 파일명: 시력이 좋아진다고 오투스 눈운동기 실제로 써본 후기_data_cleaned.json     | 📝 총 글자 수: 3,786자
📄 파일명: 와디즈 사기 기업 

In [19]:
import json
import glob
import os
import re
import pandas as pd

# ==========================================
# 1. 극한의 노이즈 커팅 함수
# ==========================================
def ultimate_noise_cutter(text):
    if not isinstance(text, str):
        return None

    text = re.sub(r'\[.*?\]|\(.*?\)|<.*?>', '', text)
    text = re.sub(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\(\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+', '', text)
    text = re.sub(r'www\.[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+', '', text)
    text = re.sub(r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+', '', text)
    text = text.strip()

    spam_phrases = [
        "구독과", "좋아요", "알림 설정", "시청해주셔서", "시청해 주셔서", "안녕하세요",
        "채널에 오신", "후원 계좌", "후원계좌", "링크를 확인", "댓글로 남겨", "문의는", "클릭해",
        "subscribe", "hit the bell", "thanks for watching", "welcome back", 
        "link in the description", "leave a comment", "smash that like"
    ]
    if any(spam in text.lower() for spam in spam_phrases):
        return None

    pure_text_length = len(re.sub(r'[\W_]+', '', text))
    if pure_text_length <= 2:
        return None

    return text

# ==========================================
# 2. 파일 비교 분석 로직
# ==========================================
# _cleaned.json이 아닌 원본 JSON 파일만 불러오기
file_list = [f for f in glob.glob("Z:/microsoft/project2/code/*.json") if "_cleaned" not in f]

comparison_data = []

print(f"총 {len(file_list)}개의 원본 파일을 분석하여 전/후를 비교합니다...\n")

for file_path in file_list:
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            
        base_name = os.path.basename(file_path)
        
        original_length = 0
        cleaned_length = 0
        
        for item in data.get('transcript', []):
            orig_text = item.get('text', '')
            original_length += len(orig_text) # 원본 글자 수 누적
            
            # 노이즈 컷팅 통과
            clean_text = ultimate_noise_cutter(orig_text)
            if clean_text:
                cleaned_length += len(clean_text) # 살아남은 글자 수 누적
                
        # 절약된 글자 수 및 감소율(%) 계산
        removed_length = original_length - cleaned_length
        reduction_rate = (removed_length / original_length * 100) if original_length > 0 else 0
        
        comparison_data.append({
            '파일명': base_name[:30] + '...' if len(base_name) > 30 else base_name, # 이름이 길면 축약
            '원본_글자수': original_length,
            '정제후_글자수': cleaned_length,
            '삭제된_노이즈': removed_length,
            '감소율(%)': round(reduction_rate, 2)
        })
            
    except Exception as e:
        print(f"❌ 에러 발생 ({file_path}): {e}")

# ==========================================
# 3. 결과 출력 (Pandas DataFrame 활용)
# ==========================================
df_compare = pd.DataFrame(comparison_data)

# 보기 좋게 표 형태로 출력
print("-" * 80)
print(df_compare.to_string(index=False))
print("-" * 80)

# 전체 요약 통계
total_orig = df_compare['원본_글자수'].sum()
total_clean = df_compare['정제후_글자수'].sum()
total_removed = df_compare['삭제된_노이즈'].sum()
total_reduction_rate = (total_removed / total_orig) * 100

print("\n🎯 [전체 요약 리포트]")
print(f"✔️ 총 원본 글자 수: {total_orig:,} 자")
print(f"✔️ 총 정제 후 글자 수: {total_clean:,} 자")
print(f"🔥 날려버린 총 노이즈: {total_removed:,} 자")
print(f"📉 평균 노이즈 감소율: {total_reduction_rate:.2f} %")

총 12개의 원본 파일을 분석하여 전/후를 비교합니다...

--------------------------------------------------------------------------------
                              파일명  원본_글자수  정제후_글자수  삭제된_노이즈  감소율(%)
Are perpetual motion machines ...    4745     4715       30    0.63
Contrails vs Chemtrails What T...    2195     2181       14    0.64
Destroying Flat Earth Without ...    5438     5438        0    0.00
Free Energy Devices Build and ...   11561    11489       72    0.62
Infinite Energy Generator with...    4487     4415       72    1.60
NASAs Secret Technology! Free ...    2147     2106       41    1.91
[바로리뷰]오투스 플러스 30일 사용 후기_data.j...    1110     1098       12    1.08
[예양육각수] 육각수의 발견 - 물에서 보물을 발견하다...    4649     4610       39    0.84
       다누리호 내가 말했째 사기라고_data.json    2979     2939       40    1.34
시력이 좋아진다고 오투스 눈운동기 실제로 써본 후기_d...    3823     3786       37    0.97
와디즈 사기 기업 참교육_육각수 샤워기 1화_data....    5817     5777       40    0.69
         평평한 지구 기초 17가지_data.json   10892    10874       18    0.17
-

In [ ]:
import re

def semantic_chunking(cleaned_transcript, max_length=800, gap_threshold=3.0):
    """
    정제된 자막 리스트를 받아 문장 종결 어미와 시간 Gap을 기준으로
    의미론적 덩어리(Chunk)로 묶어줍니다.
    
    :param cleaned_transcript: [{"start": 0.0, "duration": 2.0, "text": "..."}, ...] 형태의 리스트
    :param max_length: 한 Chunk의 최대 글자 수 (이 값을 넘으면 종결 어미에서 자름)
    :param gap_threshold: 이 시간(초) 이상 대화가 비면 화제 전환으로 간주하여 자름
    """
    
    # 1. 문장 종결을 나타내는 정규식 (다, 요, 까, 죠 등으로 끝나고 뒤에 구두점이 있거나 공백인 경우)
    # 유튜브 자막 특성상 마침표가 없는 경우가 많으므로 어미 자체를 체크
    ending_pattern = re.compile(r'(다|요|까|죠|네)[.?!]*\s*$')
    
    # 2. 문장을 이어주어야 하는 연결 어미 (고, 며, 는데, 서 등)
    connecting_pattern = re.compile(r'(고|며|는데|서|니까|지만)\s*$')

    chunks = []
    current_chunk_text = ""
    chunk_start_time = None
    last_end_time = 0.0

    for item in cleaned_transcript:
        start_time = item['start']
        # duration이 없는 데이터도 처리할 수 있도록 기본값(예: 2초) 설정
        duration = item.get('duration', 2.0) 
        text = item['text'].strip()
        
        # 첫 아이템이거나, 새 Chunk를 시작할 때 시작 시간 기록
        if chunk_start_time is None:
            chunk_start_time = start_time

        # [조건 1: 시간 Gap 기반 화제 전환 감지]
        # 이전 자막이 끝난 시간과 현재 자막 시작 시간의 차이가 gap_threshold 이상이면 분리
        # (단, 현재 텍스트가 어느 정도 모여있을 때만)
        if current_chunk_text and (start_time - last_end_time) > gap_threshold:
             chunks.append({
                "time_range": f"{chunk_start_time:.1f} - {last_end_time:.1f}",
                "text_chunk": current_chunk_text.strip()
            })
             # 초기화
             current_chunk_text = text + " "
             chunk_start_time = start_time
             last_end_time = start_time + duration
             continue

        # 텍스트 누적
        current_chunk_text += text + " "
        last_end_time = start_time + duration
        
        # [조건 2 & 3: 문장 종결 및 길이 제한]
        # 현재 누적된 글자 수가 적당히 길고(예: max_length의 70% 이상) 
        # 명확한 종결 어미로 끝났다면 여기서 Chunk를 끊음
        is_ending = ending_pattern.search(text)
        is_connecting = connecting_pattern.search(text)
        
        if is_ending and not is_connecting and len(current_chunk_text) > (max_length * 0.7):
            chunks.append({
                "time_range": f"{chunk_start_time:.1f} - {last_end_time:.1f}",
                "text_chunk": current_chunk_text.strip()
            })
            current_chunk_text = ""
            chunk_start_time = None
            
        # [조건 4: 강제 절삭]
        # 종결 어미가 계속 안 나와서 최대 길이를 넘어가 버리면 강제로 끊음
        elif len(current_chunk_text) >= max_length:
             chunks.append({
                "time_range": f"{chunk_start_time:.1f} - {last_end_time:.1f}",
                "text_chunk": current_chunk_text.strip()
            })
             current_chunk_text = ""
             chunk_start_time = None

    # 루프가 끝난 후 마지막에 남아있는 텍스트 처리
    if current_chunk_text.strip():
        # chunk_start_time이 None일 수 있으므로(마지막에 딱 떨어졌을 때) 예외 처리
        start_t = chunk_start_time if chunk_start_time is not None else start_time
        chunks.append({
            "time_range": f"{start_t:.1f} - {last_end_time:.1f}",
            "text_chunk": current_chunk_text.strip()
        })

    return chunks

## 청킹 추가

In [ ]:
import os
import json
import glob
import re
import time

# ==========================================
# 1. 극한의 노이즈 커팅 함수
# ==========================================
def ultimate_noise_cutter(text):
    if not isinstance(text, str):
        return None

    text = re.sub(r'\[.*?\]|\(.*?\)|<.*?>', '', text)
    text = re.sub(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\(\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+', '', text)
    text = re.sub(r'www\.[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+', '', text)
    text = re.sub(r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+', '', text)
    text = text.strip()

    spam_phrases = [
        "구독과", "좋아요", "알림 설정", "시청해주셔서", "시청해 주셔서", "안녕하세요",
        "채널에 오신", "후원 계좌", "후원계좌", "링크를 확인", "댓글로 남겨", "문의는", "클릭해",
        "subscribe", "hit the bell", "thanks for watching", "welcome back", 
        "link in the description", "leave a comment", "smash that like"
    ]
    if any(spam in text.lower() for spam in spam_phrases):
        return None

    pure_text_length = len(re.sub(r'[\W_]+', '', text))
    if pure_text_length <= 2:
        return None

    return text

# ==========================================
# 2. 한국어 완벽 맞춤형 청킹 함수 
# ==========================================
def korean_smart_chunking(cleaned_items, target_sentences=2, max_length=200, gap_threshold=3.0):
    # 1. 연결 패턴 (절대 안 끊음)
    connecting_pattern = re.compile(r'(은|는|이|가|고|며|는데|서|니까|지만|을|를|에|에게|에서|로|으로|의|수|것|때|줄|데|바|할|될|텐|같|지|면|보다)\s*$')

    # 2. 종결 패턴 (문장 카운트)
    ending_pattern = re.compile(r'(다|요|죠|까|네)[.?!]*(\s+|$)')
    
    # 3. 문장 시작 패턴 (접속 부사)
    transition_pattern = re.compile(r'^(그리고|그래서|그러나|하지만|그런데|그러므로|또한|즉)\b')

    chunks = []
    current_text = ""
    chunk_start_time = None
    last_end_time = 0.0
    sentence_count = 0

    for item in cleaned_items:
        start_time = item['start']
        duration = item.get('duration', 2.0)
        text = item['text'].strip()

        if chunk_start_time is None:
            chunk_start_time = start_time

        if current_text and transition_pattern.match(text):
            chunks.append({
                "time_range": f"{chunk_start_time:.1f} - {last_end_time:.1f}",
                "text_chunk": current_text.strip()
            })
            # 덩어리 초기화 후 '그리고~' 부터 새로 시작
            current_text = ""
            chunk_start_time = start_time
            sentence_count = 0

        # [규칙 1: 시간 Gap 처리]
        if current_text and (start_time - last_end_time) > gap_threshold:
            chunks.append({
                "time_range": f"{chunk_start_time:.1f} - {last_end_time:.1f}",
                "text_chunk": current_text.strip()
            })
            current_text = ""
            chunk_start_time = start_time
            sentence_count = 0

        current_text += text + " "
        last_end_time = start_time + duration
        
        # [규칙 2 & 3: 어미 기반 판별]
        found_endings = ending_pattern.findall(text)
        if found_endings:
            sentence_count += len(found_endings)

        is_connecting = connecting_pattern.search(text)

        # 2문장 도달 시 컷 (단, 연결조사로 안 끝났을 때)
        if sentence_count >= target_sentences and not is_connecting:
            chunks.append({
                "time_range": f"{chunk_start_time:.1f} - {last_end_time:.1f}",
                "text_chunk": current_text.strip()
            })
            current_text = ""
            chunk_start_time = None
            sentence_count = 0
            
        # 너무 길어질 때 강제 컷
        elif len(current_text) >= max_length and not is_connecting:
            chunks.append({
                "time_range": f"{chunk_start_time:.1f} - {last_end_time:.1f}",
                "text_chunk": current_text.strip()
            })
            current_text = ""
            chunk_start_time = None
            sentence_count = 0

    if current_text.strip():
        s_time = chunk_start_time if chunk_start_time is not None else start_time
        chunks.append({
            "time_range": f"{s_time:.1f} - {last_end_time:.1f}",
            "text_chunk": current_text.strip()
        })

    return chunks

# ==========================================
# 3. 메인 실행부
# ==========================================
def main():
    input_path = "Z:/microsoft/project2/code/*.json"
    file_list = glob.glob(input_path)
    
    target_files = [f for f in file_list if "_cleaned.json" not in f]
    
    if not target_files:
        print("💡 처리할 원본 JSON 파일이 없습니다.")
        return

    print(f"🚀 총 {len(target_files)}개의 파일 전처리를 시작합니다...\n" + "-"*50)
    
    start_time_total = time.time()
    success_count = 0

    for file_path in target_files:
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            meta = data.get('metadata', {})
            raw_transcript = data.get('transcript', [])
            
            cleaned_items = []
            for item in raw_transcript:
                clean_text = ultimate_noise_cutter(item.get("text", ""))
                if clean_text: 
                    cleaned_items.append({
                        "start": item["start"],
                        "duration": item.get("duration", 0),
                        "text": clean_text
                    })
            
            # 함수 이름과 파라미터가 완벽하게 일치하도록 수정됨!
            final_chunks = []
            if cleaned_items:
                final_chunks = korean_smart_chunking(
                    cleaned_items, 
                    target_sentences=2, 
                    max_length=200, 
                    gap_threshold=3.0
                )
            
            cleaned_data = {
                "metadata": meta,
                "chunked_transcript": final_chunks
            }
            
            dir_name = os.path.dirname(file_path)
            base_name = os.path.basename(file_path)
            file_name_only, ext = os.path.splitext(base_name)
            
            new_file_name = f"{file_name_only}_cleaned{ext}"
            new_file_path = os.path.join(dir_name, new_file_name)
            
            with open(new_file_path, 'w', encoding='utf-8') as out_f:
                json.dump(cleaned_data, out_f, ensure_ascii=False, indent=4)
                
            print(f"✅ [성공] {new_file_name} (청크 {len(final_chunks)}개)")
            success_count += 1
            
        except Exception as e:
            print(f"❌ [에러] {os.path.basename(file_path)} 처리 중 문제 발생: {e}")

    elapsed_time = time.time() - start_time_total
    print("-" * 50)
    print(f"🎉 전처리 완료! (총 {success_count}/{len(target_files)}개 성공, 소요시간: {elapsed_time:.2f}초)")


if __name__ == "__main__":
    main()

🚀 총 12개의 파일 전처리를 시작합니다...
--------------------------------------------------
✅ [성공] Are perpetual motion machines possible_data_cleaned.json (청크 22개)
✅ [성공] Contrails vs Chemtrails What Those White Streaks Really Are_data_cleaned.json (청크 11개)
✅ [성공] Destroying Flat Earth Without Using Science - Part 2 The Stars_data_cleaned.json (청크 24개)
✅ [성공] Free Energy Devices Build and Science_data_cleaned.json (청크 53개)
✅ [성공] Infinite Energy Generator with a Car Alternator220 Volts-10KW_data_cleaned.json (청크 109개)
✅ [성공] NASAs Secret Technology! Free Internet Wherever You Want, No Plan_data_cleaned.json (청크 16개)
✅ [성공] [바로리뷰]오투스 플러스 30일 사용 후기_data_cleaned.json (청크 10개)
✅ [성공] [예양육각수] 육각수의 발견 - 물에서 보물을 발견하다!_data_cleaned.json (청크 49개)
✅ [성공] 다누리호 내가 말했째 사기라고_data_cleaned.json (청크 34개)
✅ [성공] 시력이 좋아진다고 오투스 눈운동기 실제로 써본 후기_data_cleaned.json (청크 69개)
✅ [성공] 와디즈 사기 기업 참교육_육각수 샤워기 1화_data_cleaned.json (청크 69개)
✅ [성공] 평평한 지구 기초 17가지_data_cleaned.json (청크 101개)
-------------------------------------------

## 영어 추가

In [9]:
import os
import json
import glob
import re
import time

# ==========================================
# 1. 통합 노이즈 커팅 (한글/영어 스팸 모두 필터링)
# ==========================================
def ultimate_noise_cutter(text):
    if not isinstance(text, str):
        return None

    text = re.sub(r'\[.*?\]|\(.*?\)|<.*?>', '', text)
    text = re.sub(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\(\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+', '', text)
    text = re.sub(r'www\.[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+', '', text)
    text = re.sub(r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+', '', text)
    text = text.strip()

    # 한/영 스팸 키워드 통합
    spam_phrases = [
        "구독과", "좋아요", "알림 설정", "시청해주셔서", "시청해 주셔서", "안녕하세요",
        "채널에 오신", "후원 계좌", "후원계좌", "링크를 확인", "댓글로 남겨", "문의는", "클릭해",
        "subscribe", "hit the bell", "thanks for watching", "welcome back", 
        "link in the description", "leave a comment", "smash that like"
    ]
    if any(spam in text.lower() for spam in spam_phrases):
        return None

    # 의미 없는 1~2글자 찌꺼기 폐기
    pure_text_length = len(re.sub(r'[\W_]+', '', text))
    if pure_text_length <= 2:
        return None

    return text

# ==========================================
# 2-A. 한국어 맞춤형 청킹 함수 🇰🇷
# ==========================================
def korean_smart_chunking(cleaned_items, target_sentences=2, max_length=200, gap_threshold=3.0):
    connecting_pattern = re.compile(r'(은|는|이|가|고|며|는데|서|니까|지만|을|를|에|에게|에서|로|으로|의|수|것|때|줄|데|바|할|될|텐|같|지|면|보다)\s*$')
    ending_pattern = re.compile(r'(다|요|죠|까|네)[.?!]*(\s+|$)')
    transition_pattern = re.compile(r'^(그리고|그래서|그러나|하지만|그런데|그러므로|또한|즉)\b')

    chunks = []
    current_text = ""
    chunk_start_time = None
    last_end_time = 0.0
    sentence_count = 0

    for item in cleaned_items:
        start_time = item['start']
        duration = item.get('duration', 2.0)
        text = item['text'].strip()

        if chunk_start_time is None: chunk_start_time = start_time

        if current_text and transition_pattern.match(text):
            chunks.append({"time_range": f"{chunk_start_time:.1f} - {last_end_time:.1f}", "text_chunk": current_text.strip()})
            current_text = ""
            chunk_start_time = start_time
            sentence_count = 0

        if current_text and (start_time - last_end_time) > gap_threshold:
            chunks.append({"time_range": f"{chunk_start_time:.1f} - {last_end_time:.1f}", "text_chunk": current_text.strip()})
            current_text = ""
            chunk_start_time = start_time
            sentence_count = 0

        current_text += text + " "
        last_end_time = start_time + duration
        
        found_endings = ending_pattern.findall(text)
        if found_endings: sentence_count += len(found_endings)
        is_connecting = connecting_pattern.search(text)

        if sentence_count >= target_sentences and not is_connecting:
            chunks.append({"time_range": f"{chunk_start_time:.1f} - {last_end_time:.1f}", "text_chunk": current_text.strip()})
            current_text, chunk_start_time, sentence_count = "", None, 0
            
        elif len(current_text) >= max_length and not is_connecting:
            chunks.append({"time_range": f"{chunk_start_time:.1f} - {last_end_time:.1f}", "text_chunk": current_text.strip()})
            current_text, chunk_start_time, sentence_count = "", None, 0

    if current_text.strip():
        s_time = chunk_start_time if chunk_start_time is not None else start_time
        chunks.append({"time_range": f"{s_time:.1f} - {last_end_time:.1f}", "text_chunk": current_text.strip()})
    return chunks

# ==========================================
# 2-B. 영어 맞춤형 청킹 함수 🇺🇸
# ==========================================
def english_smart_chunking(cleaned_items, target_sentences=2, min_length=120, max_length=250, gap_threshold=3.0):
    # 1. 절대 끊지 않는 패턴 (전치사, be동사, and/but 등 접속사 유지)
    connecting_pattern = re.compile(r'\b(and|or|but|because|if|since|that|which|who|with|to|in|on|at|by|for|of|the|a|an|is|are|was|were|will|would|can|could|should|be|as|than)\s*$', re.IGNORECASE)
    
    # 2. 종결 패턴 (마침표가 있는 고급 자막용)
    ending_pattern = re.compile(r'[.?!]+(\s+|$)')
    
    # 3. 강제 절단 패턴 (의미가 완전히 뒤집히는 강한 부사만 남김)
    transition_pattern = re.compile(r'^(however|therefore|moreover|thus|meanwhile|additionally|furthermore|nevertheless)\b', re.IGNORECASE)

    # 💡 4. [NEW] 소프트 바운더리 패턴 (STT 무구두점 대비용)
    # 다음 자막이 대명사나 가벼운 부사로 시작하면 새로운 문장의 시작으로 간주!
    clause_starter_pattern = re.compile(r'^(i|you|he|she|we|they|it|this|that|there|here|so|now|well|then|anyway)\b', re.IGNORECASE)

    chunks = []
    current_text = ""
    chunk_start_time = None
    last_end_time = 0.0
    sentence_count = 0

    for item in cleaned_items:
        start_time = item['start']
        duration = item.get('duration', 2.0)
        text = item['text'].strip()

        if chunk_start_time is None: 
            chunk_start_time = start_time

        # [규칙 1: 강한 부사 컷] - However 등이 나오면 즉시 앞문맥 컷
        if current_text and transition_pattern.match(text):
            chunks.append({"time_range": f"{chunk_start_time:.1f} - {last_end_time:.1f}", "text_chunk": current_text.strip()})
            current_text, chunk_start_time, sentence_count = "", start_time, 0

        # [규칙 2: 시간 Gap 컷] - 3초 이상 침묵 시 컷
        if current_text and (start_time - last_end_time) > gap_threshold:
            chunks.append({"time_range": f"{chunk_start_time:.1f} - {last_end_time:.1f}", "text_chunk": current_text.strip()})
            current_text, chunk_start_time, sentence_count = "", start_time, 0

        # 💡 [NEW 규칙 3: 무구두점 대비 소프트 바운더리 컷]
        if len(current_text) >= min_length and clause_starter_pattern.match(text) and not connecting_pattern.search(current_text):
            chunks.append({"time_range": f"{chunk_start_time:.1f} - {last_end_time:.1f}", "text_chunk": current_text.strip()})
            current_text, chunk_start_time, sentence_count = "", start_time, 0

        current_text += text + " "
        last_end_time = start_time + duration
        
        # [규칙 4: 마침표 기반 정상 카운트 컷] (마침표가 있는 자막일 경우 작동)
        found_endings = ending_pattern.findall(text)
        if found_endings: 
            sentence_count += len(found_endings)
            
        is_connecting = connecting_pattern.search(text)

        if sentence_count >= target_sentences and not is_connecting:
            chunks.append({"time_range": f"{chunk_start_time:.1f} - {last_end_time:.1f}", "text_chunk": current_text.strip()})
            current_text, chunk_start_time, sentence_count = "", None, 0
            
        # [규칙 5: 최대 길이 방어막] (안전장치)
        elif len(current_text) >= max_length and not is_connecting:
            chunks.append({"time_range": f"{chunk_start_time:.1f} - {last_end_time:.1f}", "text_chunk": current_text.strip()})
            current_text, chunk_start_time, sentence_count = "", None, 0

    # 찌꺼기 털어내기
    if current_text.strip():
        s_time = chunk_start_time if chunk_start_time is not None else start_time
        chunks.append({"time_range": f"{s_time:.1f} - {last_end_time:.1f}", "text_chunk": current_text.strip()})
        
    return chunks
# ==========================================
# 3. 언어 자동 감지 함수 (Auto Language Detector)
# ==========================================
def detect_language(raw_transcript):
    """
    영상 초반 자막 데이터를 분석하여 한국어/영어를 판별합니다.
    """
    if not raw_transcript:
        return "EN" # 비어있으면 기본값 영어
        
    # 처음 20개 자막 덩어리를 모아서 샘플 텍스트 생성
    sample_text = "".join([item.get("text", "") for item in raw_transcript[:20]])
    
    # 한글([가-힣])이 샘플 안에 5글자 이상 있으면 한국어 영상으로 간주
    hangul_count = len(re.findall(r'[가-힣]', sample_text))
    
    if hangul_count >= 5:
        return "KO"
    else:
        return "EN"

# ==========================================
# 4. 메인 실행부 (스마트 라우팅)
# ==========================================
def main():
    input_path = "Z:/microsoft/project2/code/*.json"
    file_list = glob.glob(input_path)
    
    target_files = [f for f in file_list if "_cleaned.json" not in f]
    
    if not target_files:
        print("💡 처리할 원본 JSON 파일이 없습니다.")
        return

    print(f"🚀 총 {len(target_files)}개의 파일 전처리를 시작합니다. (언어 자동 인식 탑재)\n" + "-"*60)
    
    start_time_total = time.time()
    success_count = 0

    for file_path in target_files:
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            meta = data.get('metadata', {})
            raw_transcript = data.get('transcript', [])
            
            # 💡 1. 언어 자동 감지
            detected_lang = detect_language(raw_transcript)
            lang_flag = "🇰🇷 한국어" if detected_lang == "KO" else "🇺🇸 영어"
            
            # 💡 2. 노이즈 제거 (공통)
            cleaned_items = []
            for item in raw_transcript:
                clean_text = ultimate_noise_cutter(item.get("text", ""))
                if clean_text: 
                    cleaned_items.append({
                        "start": item["start"],
                        "duration": item.get("duration", 0),
                        "text": clean_text
                    })
            
            # 💡 3. 감지된 언어에 맞춰 청킹 엔진 라우팅!
            final_chunks = []
            if cleaned_items:
                if detected_lang == "KO":
                    final_chunks = korean_smart_chunking(cleaned_items, target_sentences=2, max_length=200, gap_threshold=3.0)
                else:
                    final_chunks = english_smart_chunking(cleaned_items, target_sentences=2, max_length=250, gap_threshold=3.0)
            
            cleaned_data = {
                "metadata": meta,
                "detected_language": detected_lang, # 메타데이터에 판별된 언어도 기록해 둡니다.
                "chunked_transcript": final_chunks
            }
            
            dir_name = os.path.dirname(file_path)
            base_name = os.path.basename(file_path)
            file_name_only, ext = os.path.splitext(base_name)
            
            new_file_name = f"{file_name_only}_cleaned{ext}"
            new_file_path = os.path.join(dir_name, new_file_name)
            
            with open(new_file_path, 'w', encoding='utf-8') as out_f:
                json.dump(cleaned_data, out_f, ensure_ascii=False, indent=4)
                
            print(f"✅ [성공 | {lang_flag}] {new_file_name} (청크 {len(final_chunks)}개)")
            success_count += 1
            
        except Exception as e:
            print(f"❌ [에러] {os.path.basename(file_path)} 처리 중 문제 발생: {e}")

    elapsed_time = time.time() - start_time_total
    print("-" * 60)
    print(f"🎉 통합 전처리 완료! (총 {success_count}/{len(target_files)}개 성공, 소요시간: {elapsed_time:.2f}초)")

if __name__ == "__main__":
    main()

🚀 총 12개의 파일 전처리를 시작합니다. (언어 자동 인식 탑재)
------------------------------------------------------------
✅ [성공 | 🇺🇸 영어] Are perpetual motion machines possible_data_cleaned.json (청크 32개)
✅ [성공 | 🇺🇸 영어] Contrails vs Chemtrails What Those White Streaks Really Are_data_cleaned.json (청크 9개)
✅ [성공 | 🇺🇸 영어] Destroying Flat Earth Without Using Science - Part 2 The Stars_data_cleaned.json (청크 34개)
✅ [성공 | 🇺🇸 영어] Free Energy Devices Build and Science_data_cleaned.json (청크 47개)
✅ [성공 | 🇰🇷 한국어] Infinite Energy Generator with a Car Alternator220 Volts-10KW_data_cleaned.json (청크 109개)
✅ [성공 | 🇰🇷 한국어] NASAs Secret Technology! Free Internet Wherever You Want, No Plan_data_cleaned.json (청크 16개)
✅ [성공 | 🇰🇷 한국어] [바로리뷰]오투스 플러스 30일 사용 후기_data_cleaned.json (청크 10개)
✅ [성공 | 🇰🇷 한국어] [예양육각수] 육각수의 발견 - 물에서 보물을 발견하다!_data_cleaned.json (청크 49개)
✅ [성공 | 🇰🇷 한국어] 다누리호 내가 말했째 사기라고_data_cleaned.json (청크 34개)
✅ [성공 | 🇰🇷 한국어] 시력이 좋아진다고 오투스 눈운동기 실제로 써본 후기_data_cleaned.json (청크 69개)
✅ [성공 | 🇰🇷 한국어] 와디즈 사기 기업 참교육_육각수 샤워기 1화_data

In [14]:
import json
import re
import os
import glob

# ==========================================
# 1. 극한의 노이즈 커팅
# ==========================================
def ultimate_noise_cutter(text):
    if not isinstance(text, str): return None
    text = re.sub(r'\[.*?\]|\(.*?\)|<.*?>', '', text)
    text = re.sub(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\(\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+', '', text)
    text = re.sub(r'www\.[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+', '', text)
    text = re.sub(r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+', '', text)
    text = text.strip()

    spam_phrases = [
        "구독과", "좋아요", "알림 설정", "시청해주셔서", "시청해 주셔서", "안녕하세요",
        "채널에 오신", "후원 계좌", "후원계좌", "링크를 확인", "댓글로 남겨", "문의는", "클릭해",
        "subscribe", "hit the bell", "thanks for watching", "welcome back", 
        "link in the description", "leave a comment", "smash that like"
    ]
    if any(spam in text.lower() for spam in spam_phrases): return None
    
    if len(re.sub(r'[\W_]+', '', text)) <= 2: return None
    return text

# ==========================================
# 2. 언어 자동 감지
# ==========================================
def detect_language(raw_transcript):
    if not raw_transcript: return "en"
    sample_text = "".join([item.get("text", "") for item in raw_transcript[:20]])
    return "ko" if len(re.findall(r'[가-힣]', sample_text)) >= 5 else "en"

# ==========================================
# 3. 메인 실행부 (단순 병합)
# ==========================================
def main():
    input_path = "Z:/microsoft/project2/code/*.json"
    file_list = glob.glob(input_path)
    
    # 처리 제외: 이미 합쳐진 파일이나, 예전 청킹 포맷 파일
    target_files = [f for f in file_list if "_merged.json" not in f and "_cleaned.json" not in f]
    
    if not target_files:
        print("💡 처리할 원본 JSON 파일이 없습니다.")
        return

    print(f"🚀 총 {len(target_files)}개의 파일 처리 시작 (단순 병합)")
    print("-" * 50)
    
    for file_path in target_files:
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            meta = data.get('metadata', {})
            raw_transcript = data.get('transcript', [])
            
            # 1. 메타데이터 및 언어 감지
            lang = detect_language(raw_transcript)
            raw_title = meta.get("title", "Unknown_Title")
            safe_title = re.sub(r'[\\/*?:"<>|]', "", raw_title).replace(" ", "_")
            source = meta.get("source_url", f"https://www.youtube.com/watch?v={meta.get('video_id', '')}")
            
            # 2. 노이즈 컷팅 및 단순 병합 (띄어쓰기로 연결)
            cleaned_texts = []
            for item in raw_transcript:
                clean_text = ultimate_noise_cutter(item.get("text", ""))
                if clean_text: 
                    cleaned_texts.append(clean_text)
            
            merged_script = " ".join(cleaned_texts)
            
            # 3. YAML 프론트매터 템플릿 생성
            formatted_content = f"""---
title: "{raw_title}"
source: "{source}"
language: "{lang}"
word_count: {len(merged_script)}
---

{merged_script}"""

            output_data = {
                "transcript": formatted_content
            }
            
            # 4. 파일 저장
            dir_name = os.path.dirname(file_path)
            output_filename = os.path.join(dir_name, f"{safe_title}_merged.json")
            
            with open(output_filename, 'w', encoding='utf-8') as out_f:
                json.dump(output_data, out_f, ensure_ascii=False, indent=4)
            
            print(f"✅ 생성 완료: {safe_title}_merged.json")
            
        except Exception as e:
            print(f"❌ 에러 발생 ({os.path.basename(file_path)}): {e}")

if __name__ == "__main__":
    main()

🚀 총 12개의 파일 처리 시작 (단순 병합)
--------------------------------------------------
✅ 생성 완료: Are_perpetual_motion_machines_possible_merged.json
✅ 생성 완료: Contrails_vs_“Chemtrails”_What_Those_White_Streaks_Really_Are_merged.json
✅ 생성 완료: Destroying_Flat_Earth_Without_Using_Science_-_Part_2_The_Stars_merged.json
✅ 생성 완료: Free_Energy_Devices_Build_and_Science_merged.json
✅ 생성 완료: Infinite_Energy_Generator_with_a_Car_Alternator🔋⚡220_Volts-10KW_merged.json
✅ 생성 완료: NASA's_Secret_Technology!_Free_Internet_Wherever_You_Want,_No_Plan_merged.json
✅ 생성 완료: [바로리뷰]오투스_플러스_30일_사용_후기_merged.json
✅ 생성 완료: [예양육각수]_육각수의_발견_-_물에서_보물을_발견하다!_merged.json
✅ 생성 완료: 다누리호_내가_말했째__사기라고_merged.json
✅ 생성 완료: 시력이_좋아진다고_오투스_눈운동기_실제로_써본_후기_merged.json
✅ 생성 완료: 와디즈_사기_기업_참교육_육각수_샤워기_#1화_merged.json
✅ 생성 완료: 평평한_지구_기초_17가지_merged.json


## 줄바꿈 버전

In [16]:
import os
import json
import glob
import re
import time

# ==========================================
# 1. 통합 노이즈 커팅 (한글/영어 스팸 모두 필터링)
# ==========================================
def ultimate_noise_cutter(text):
    if not isinstance(text, str):
        return None

    text = re.sub(r'\[.*?\]|\(.*?\)|<.*?>', '', text)
    text = re.sub(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\(\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+', '', text)
    text = re.sub(r'www\.[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+', '', text)
    text = re.sub(r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+', '', text)
    text = text.strip()

    # 한/영 스팸 키워드 통합
    spam_phrases = [
        "구독과", "좋아요", "알림 설정", "시청해주셔서", "시청해 주셔서", "안녕하세요",
        "채널에 오신", "후원 계좌", "후원계좌", "링크를 확인", "댓글로 남겨", "문의는", "클릭해",
        "subscribe", "hit the bell", "thanks for watching", "welcome back", 
        "link in the description", "leave a comment", "smash that like"
    ]
    if any(spam in text.lower() for spam in spam_phrases):
        return None

    # 의미 없는 1~2글자 찌꺼기 폐기
    pure_text_length = len(re.sub(r'[\W_]+', '', text))
    if pure_text_length <= 2:
        return None

    return text

# ==========================================
# 2-A. 한국어 맞춤형 청킹 함수 🇰🇷 (원본 유지)
# ==========================================
def korean_smart_chunking(cleaned_items, target_sentences=2, max_length=200, gap_threshold=3.0):
    connecting_pattern = re.compile(r'(은|는|이|가|고|며|는데|서|니까|지만|을|를|에|에게|에서|로|으로|의|수|것|때|줄|데|바|할|될|텐|같|지|면|보다)\s*$')
    ending_pattern = re.compile(r'(다|요|죠|까|네)[.?!]*(\s+|$)')
    transition_pattern = re.compile(r'^(그리고|그래서|그러나|하지만|그런데|그러므로|또한|즉)\b')

    chunks = []
    current_text = ""
    chunk_start_time = None
    last_end_time = 0.0
    sentence_count = 0

    for item in cleaned_items:
        start_time = item['start']
        duration = item.get('duration', 2.0)
        text = item['text'].strip()

        if chunk_start_time is None: chunk_start_time = start_time

        if current_text and transition_pattern.match(text):
            chunks.append({"time_range": f"{chunk_start_time:.1f} - {last_end_time:.1f}", "text_chunk": current_text.strip()})
            current_text = ""
            chunk_start_time = start_time
            sentence_count = 0

        if current_text and (start_time - last_end_time) > gap_threshold:
            chunks.append({"time_range": f"{chunk_start_time:.1f} - {last_end_time:.1f}", "text_chunk": current_text.strip()})
            current_text = ""
            chunk_start_time = start_time
            sentence_count = 0

        current_text += text + " "
        last_end_time = start_time + duration
        
        found_endings = ending_pattern.findall(text)
        if found_endings: sentence_count += len(found_endings)
        is_connecting = connecting_pattern.search(text)

        if sentence_count >= target_sentences and not is_connecting:
            chunks.append({"time_range": f"{chunk_start_time:.1f} - {last_end_time:.1f}", "text_chunk": current_text.strip()})
            current_text, chunk_start_time, sentence_count = "", None, 0
            
        elif len(current_text) >= max_length and not is_connecting:
            chunks.append({"time_range": f"{chunk_start_time:.1f} - {last_end_time:.1f}", "text_chunk": current_text.strip()})
            current_text, chunk_start_time, sentence_count = "", None, 0

    if current_text.strip():
        s_time = chunk_start_time if chunk_start_time is not None else start_time
        chunks.append({"time_range": f"{s_time:.1f} - {last_end_time:.1f}", "text_chunk": current_text.strip()})
    return chunks

# ==========================================
# 2-B. 영어 맞춤형 청킹 함수 🇺🇸 (원본 유지)
# ==========================================
def english_smart_chunking(cleaned_items, target_sentences=2, min_length=120, max_length=250, gap_threshold=3.0):
    connecting_pattern = re.compile(r'\b(and|or|but|because|if|since|that|which|who|with|to|in|on|at|by|for|of|the|a|an|is|are|was|were|will|would|can|could|should|be|as|than)\s*$', re.IGNORECASE)
    ending_pattern = re.compile(r'[.?!]+(\s+|$)')
    transition_pattern = re.compile(r'^(however|therefore|moreover|thus|meanwhile|additionally|furthermore|nevertheless)\b', re.IGNORECASE)
    clause_starter_pattern = re.compile(r'^(i|you|he|she|we|they|it|this|that|there|here|so|now|well|then|anyway)\b', re.IGNORECASE)

    chunks = []
    current_text = ""
    chunk_start_time = None
    last_end_time = 0.0
    sentence_count = 0

    for item in cleaned_items:
        start_time = item['start']
        duration = item.get('duration', 2.0)
        text = item['text'].strip()

        if chunk_start_time is None: 
            chunk_start_time = start_time

        if current_text and transition_pattern.match(text):
            chunks.append({"time_range": f"{chunk_start_time:.1f} - {last_end_time:.1f}", "text_chunk": current_text.strip()})
            current_text, chunk_start_time, sentence_count = "", start_time, 0

        if current_text and (start_time - last_end_time) > gap_threshold:
            chunks.append({"time_range": f"{chunk_start_time:.1f} - {last_end_time:.1f}", "text_chunk": current_text.strip()})
            current_text, chunk_start_time, sentence_count = "", start_time, 0

        if len(current_text) >= min_length and clause_starter_pattern.match(text) and not connecting_pattern.search(current_text):
            chunks.append({"time_range": f"{chunk_start_time:.1f} - {last_end_time:.1f}", "text_chunk": current_text.strip()})
            current_text, chunk_start_time, sentence_count = "", start_time, 0

        current_text += text + " "
        last_end_time = start_time + duration
        
        found_endings = ending_pattern.findall(text)
        if found_endings: 
            sentence_count += len(found_endings)
            
        is_connecting = connecting_pattern.search(text)

        if sentence_count >= target_sentences and not is_connecting:
            chunks.append({"time_range": f"{chunk_start_time:.1f} - {last_end_time:.1f}", "text_chunk": current_text.strip()})
            current_text, chunk_start_time, sentence_count = "", None, 0
            
        elif len(current_text) >= max_length and not is_connecting:
            chunks.append({"time_range": f"{chunk_start_time:.1f} - {last_end_time:.1f}", "text_chunk": current_text.strip()})
            current_text, chunk_start_time, sentence_count = "", None, 0

    if current_text.strip():
        s_time = chunk_start_time if chunk_start_time is not None else start_time
        chunks.append({"time_range": f"{s_time:.1f} - {last_end_time:.1f}", "text_chunk": current_text.strip()})
        
    return chunks

# ==========================================
# 3. 언어 자동 감지 함수
# ==========================================
def detect_language(raw_transcript):
    if not raw_transcript:
        return "EN"
    sample_text = "".join([item.get("text", "") for item in raw_transcript[:20]])
    hangul_count = len(re.findall(r'[가-힣]', sample_text))
    if hangul_count >= 5:
        return "KO"
    else:
        return "EN"

# ==========================================
# 4. 메인 실행부 (스마트 라우팅 및 병합)
# ==========================================
def main():
    input_path = "Z:/microsoft/project2/code/*.json"
    file_list = glob.glob(input_path)
    
    # 💡 대상 파일 필터링: 원본 JSON만 골라내기
    target_files = [f for f in file_list if "_merged.json" not in f and "_cleaned.json" not in f]
    
    if not target_files:
        print("💡 처리할 원본 JSON 파일이 없습니다.")
        return

    print(f"🚀 총 {len(target_files)}개의 파일 전처리를 시작합니다. (청킹 후 줄바꿈 병합)\n" + "-"*60)
    
    start_time_total = time.time()
    success_count = 0

    for file_path in target_files:
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            meta = data.get('metadata', {})
            raw_transcript = data.get('transcript', [])
            
            # 1. 언어 자동 감지 및 메타데이터 추출
            detected_lang = detect_language(raw_transcript)
            lang_flag = "🇰🇷 한국어" if detected_lang == "KO" else "🇺🇸 영어"
            
            raw_title = meta.get("title", "Unknown_Title")
            # 💡 안전한 파일명 생성 (특수문자 제거, 공백은 언더바로)
            safe_title = re.sub(r'[\\/*?:"<>|]', "", raw_title).replace(" ", "_")
            source = meta.get("source_url", f"https://www.youtube.com/watch?v={meta.get('video_id', '')}")
            
            # 2. 노이즈 제거
            cleaned_items = []
            for item in raw_transcript:
                clean_text = ultimate_noise_cutter(item.get("text", ""))
                if clean_text: 
                    cleaned_items.append({
                        "start": item["start"],
                        "duration": item.get("duration", 0),
                        "text": clean_text
                    })
            
            # 3. 감지된 언어에 맞춰 청킹 엔진 라우팅!
            final_chunks = []
            if cleaned_items:
                if detected_lang == "KO":
                    final_chunks = korean_smart_chunking(cleaned_items, target_sentences=2, max_length=200, gap_threshold=3.0)
                else:
                    final_chunks = english_smart_chunking(cleaned_items, target_sentences=2, min_length=120, max_length=250, gap_threshold=3.0)
            
            # 💡 4. 핵심 병합 로직: 청킹된 결과물의 'text_chunk'만 뽑아서 줄바꿈(\n\n)으로 병합
            merged_script = "\n\n".join([chunk["text_chunk"] for chunk in final_chunks])
            
            # 💡 5. LLM용 YAML 프론트매터 템플릿 생성
            formatted_content = f"""---
title: "{raw_title}"
source: "{source}"
language: "{detected_lang}"
word_count: {len(merged_script)}
---

{merged_script}"""

            # 최종 JSON 구조
            output_data = {
                "transcript": formatted_content
            }
            
            # 6. 파일 저장 (제목_merged.json 형식)
            dir_name = os.path.dirname(file_path)
            new_file_name = f"{safe_title}_merged.json"
            new_file_path = os.path.join(dir_name, new_file_name)
            
            with open(new_file_path, 'w', encoding='utf-8') as out_f:
                json.dump(output_data, out_f, ensure_ascii=False, indent=4)
                
            print(f"✅ [성공 | {lang_flag}] {new_file_name}")
            success_count += 1
            
        except Exception as e:
            print(f"❌ [에러] {os.path.basename(file_path)} 처리 중 문제 발생: {e}")

    elapsed_time = time.time() - start_time_total
    print("-" * 60)
    print(f"🎉 통합 전처리 완료! (총 {success_count}/{len(target_files)}개 성공, 소요시간: {elapsed_time:.2f}초)")

if __name__ == "__main__":
    main()

💡 처리할 원본 JSON 파일이 없습니다.
